# 07 - Pipeline `vit-lora` (ViT-Base + LoRA, parameter-efficient fine-tuning)

Fine-tune a frozen ImageNet-21k **ViT-Base** at 224px using **LoRA** on the attention `qkv` projections (+ a trainable head). Only ~0.7% of parameters train, checkpoints are tiny, and at inference the LoRA update is merged into the backbone (zero added latency).

**Sections:** 0 Setup · 1 Data · 2 Model (LoRA) · 3 Training setup · 4 Train (visible loop) · 5 Curves · 6 Merge + in-distribution eval · 7 Cross-generator (OOD) preview · 8 Attention rollout · 9 Save metrics.json.

Requires `03_split_and_preprocessing`. Artifacts → `notebooks/artifacts/vit-lora/{models,figures,metrics}`.

## 0 - Setup

In [ ]:
import sys, time
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from IPython.display import display

_here = Path.cwd()
_nb_dir = _here if (_here / "utils").is_dir() else _here / "notebooks"
if str(_nb_dir) not in sys.path:
    sys.path.insert(0, str(_nb_dir))

from utils import datasets as D, models as M, training as T, metrics as Me, viz as V, explain as E, eda
from utils.paths import repo_paths, artifact_dirs

torch.manual_seed(42); np.random.seed(42)
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.set_float32_matmul_precision("high")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

PATHS = repo_paths(_nb_dir)
DATA_DIR = PATHS["data"]; AIR_DIR = DATA_DIR / "ai-real-images"
SPLIT_PATH = AIR_DIR / "manifest_split.csv"
TINY_MANIFEST = DATA_DIR / "tiny-genimage" / "manifest_clean.csv"

PIPELINE = "vit-lora"
WORKING_SIZE = 224
NORM = "imagenet"
BATCH_SIZE = 64
EPOCHS = 12
WARMUP_EPOCHS = 1
LABEL_SMOOTH = 0.05
NUM_WORKERS = 8
dirs = artifact_dirs(PIPELINE)
print("device:", device, "| pipeline:", PIPELINE)

## 1 - Data

224px / ImageNet-norm loaders (the ViT was pretrained with ImageNet stats).

In [ ]:
loaders = D.make_loaders(SPLIT_PATH, working_size=WORKING_SIZE, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, norm=NORM)
train_loader, val_loader, test_loader = loaders["train"], loaders["val"], loaders["test"]
mean, std = D.resolve_stats(NORM, AIR_DIR)
split_df = pd.read_csv(SPLIT_PATH); split_df = split_df[split_df["keep"]]
test_df = split_df[split_df["split_final"] == "test"].reset_index(drop=True)
print(f"train {len(train_loader.dataset):,} | val {len(val_loader.dataset):,} | test {len(test_loader.dataset):,}")

## 2 - Model (ViT-Base + LoRA)

Frozen `vit_base_patch16_224.augreg_in21k`; LoRA (rank 16) injected into the attention `qkv` projections; the classification head is also trainable (`modules_to_save`). The first run downloads the ViT-21k weights (~330 MB).

In [ ]:
model = M.build_vit_lora(r=16, lora_alpha=16, lora_dropout=0.05).to(device)
tr = sum(p.numel() for p in model.parameters() if p.requires_grad)
tot = sum(p.numel() for p in model.parameters())
print(f"trainable {tr:,} / {tot:,}  ({100*tr/tot:.2f}%)")
with torch.no_grad():
    print("dummy forward:", tuple(model(torch.randn(2, 3, WORKING_SIZE, WORKING_SIZE, device=device)).shape))

## 3 - Training setup

Only LoRA + head params receive gradients. AdamW + cosine warmup, early-stop on val AUC.

In [ ]:
loss_fn = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=1e-3, weight_decay=1e-4)
spe = len(train_loader)
scheduler = T.build_cosine_with_warmup(optimizer, total_steps=EPOCHS * spe, warmup_steps=WARMUP_EPOCHS * spe)
stopper = T.EarlyStopper(mode="max", patience=5, min_delta=1e-3)
history = {"train_loss": [], "val_loss": [], "val_auc": [], "val_acc": []}
best_auc = -1.0
ckpt_path = dirs["models"] / "best.pt"

## 4 - Train (visible loop)

In [ ]:
for epoch in range(EPOCHS):
    t0 = time.time()
    tr_m = T.train_one_epoch(model, train_loader, optimizer, loss_fn, device, scheduler=scheduler, label_smooth=LABEL_SMOOTH)
    yv, pv, vloss = T.evaluate(model, val_loader, device, loss_fn)
    vm = Me.classification_metrics(yv, pv)
    history["train_loss"].append(tr_m["loss"]); history["val_loss"].append(vloss)
    history["val_auc"].append(vm["auc_roc"]); history["val_acc"].append(vm["accuracy"])
    improved, stop = stopper.step(vm["auc_roc"])
    if improved:
        best_auc = vm["auc_roc"]
        # Save ONLY the trained part (LoRA + head) -> tiny, GitHub-friendly.
        # Rebuild with M.build_vit_lora() (downloads the frozen ViT) + load_weights(strict=False).
        T.save_weights(ckpt_path, T.trained_state_dict(model),
                       meta={"pipeline": PIPELINE, "kind": "lora+head", "epoch": epoch, "val_auc": best_auc})
    print(f"epoch {epoch+1:02d} | train_loss {tr_m['loss']:.4f} | val_loss {vloss:.4f} | val_auc {vm['auc_roc']:.4f} | val_acc {vm['accuracy']:.4f} | {time.time()-t0:.0f}s{'  *best' if improved else ''}")
    if stop:
        print("early stopping"); break

## 5 - Training curves

In [ ]:
V.plot_training_curves(history).savefig(dirs["figures"] / "training_curves.png", dpi=150, bbox_inches="tight")
plt.show()

## 6 - Merge LoRA + in-distribution evaluation

Load the best checkpoint, **merge** the LoRA weights into the backbone (`merge_and_unload`) to get a plain ViT, then evaluate on the test set at 0.5 and a val-tuned threshold.

In [ ]:
T.load_weights(ckpt_path, model, strict=False, map_location=device)   # load trained LoRA+head into the peft model
model = model.merge_and_unload()      # -> plain timm ViT with LoRA merged in
model.eval()
yt, pt, _ = T.evaluate(model, test_loader, device)
yv, pv, _ = T.evaluate(model, val_loader, device)
tuned = Me.best_f1_threshold(yv, pv)
m05 = Me.classification_metrics(yt, pt, threshold=0.5)
mtuned = Me.classification_metrics(yt, pt, threshold=tuned["threshold"])
print("tuned threshold (on val):", round(tuned["threshold"], 4))
display(Me.summary_table(m05))
V.plot_confusion(m05["confusion_matrix"]).savefig(dirs["figures"] / "confusion.png", dpi=150, bbox_inches="tight")
V.plot_roc_pr(yt, pt).savefig(dirs["figures"] / "roc_pr.png", dpi=150, bbox_inches="tight")
V.plot_reliability(yt, pt).savefig(dirs["figures"] / "reliability.png", dpi=150, bbox_inches="tight")
plt.show()

## 7 - Cross-generator (OOD) preview — tiny-genimage

In [ ]:
GEN_MAP = {
    "imagenet_ai_0419_biggan": "biggan", "imagenet_ai_0419_vqdm": "vqdm",
    "imagenet_ai_0424_sdv5": "sdv5", "imagenet_ai_0424_wukong": "wukong",
    "imagenet_ai_0508_adm": "adm", "imagenet_glide": "glide", "imagenet_midjourney": "midjourney",
}
ood_loader, ood_df = D.make_ood_loader(TINY_MANIFEST, WORKING_SIZE, BATCH_SIZE, mean, std, num_workers=NUM_WORKERS)
yo, po, _ = T.evaluate(model, ood_loader, device)
ood_df = ood_df.assign(p_fake=po, y_true=yo)
ood_df["y_pred"] = (ood_df["p_fake"] >= 0.5).astype(int)
ood_df["generator"] = ood_df["source"].map(GEN_MAP).fillna(ood_df["source"])
rows = [{"generator": gen, "accuracy": float((g["y_pred"] == g["y_true"]).mean()), "n": int(len(g))}
        for gen, g in ood_df.groupby("generator")]
per_gen = pd.DataFrame(rows)
overall_ood = float((ood_df["y_pred"] == ood_df["y_true"]).mean())
display(per_gen)
print(f"overall OOD accuracy: {overall_ood:.4f}  (in-dist test acc: {m05['accuracy']:.4f})")
V.plot_per_generator_bar(per_gen, ref_acc=m05["accuracy"]).savefig(dirs["figures"] / "ood_per_generator.png", dpi=150, bbox_inches="tight")
plt.show()

## 8 - Explainability (attention rollout)

For ViTs we use **attention rollout** (Abnar & Zuidema) instead of Grad-CAM: it multiplies the per-layer attention maps (with residual identity) to show which patches drive the `[CLS]` decision.

In [ ]:
from pytorch_grad_cam.utils.image import show_cam_on_image
eval_tf = D.build_eval_tf(WORKING_SIZE, mean, std)
examples = E.pick_examples(test_df, n_per_class=3, seed=0)
fig, axes = plt.subplots(2, len(examples), figsize=(2.2 * len(examples), 4.6))
for j, ex in enumerate(examples):
    arr = eda.read_rgb(ex["filepath"])
    xt = eval_tf(torch.from_numpy(arr).permute(2, 0, 1))
    x = xt.unsqueeze(0).to(device)
    rgb = D.denormalize(xt, mean, std).permute(1, 2, 0).numpy()
    with torch.no_grad():
        p = torch.sigmoid(model(x)).item()
    mask = E.attention_rollout(model, x, out_size=WORKING_SIZE)
    overlay = show_cam_on_image(np.ascontiguousarray(rgb, dtype=np.float32), mask, use_rgb=True)
    axes[0, j].imshow(rgb); axes[0, j].axis("off"); axes[0, j].set_title(f"{ex['label']}  p_fake={p:.2f}", fontsize=8)
    axes[1, j].imshow(overlay); axes[1, j].axis("off")
fig.suptitle("vit-lora attention rollout", fontsize=11)
fig.savefig(dirs["figures"] / "attention_rollout.png", dpi=150, bbox_inches="tight")
plt.show()

## 9 - Save metrics.json

In [ ]:
record = {
    "pipeline": PIPELINE,
    "created": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "working_size": WORKING_SIZE, "normalization": NORM,
    "dataset": {"in_distribution": "ai-real-images", "ood": "tiny-genimage"},
    "threshold_default": 0.5, "threshold_tuned": tuned["threshold"],
    "in_distribution": {"at_0.5": m05, "at_tuned": mtuned},
    "ood": {"overall_accuracy": overall_ood,
             "per_generator": {r.generator: {"accuracy": r.accuracy, "n": r.n} for r in per_gen.itertuples()},
             "preview": True},
    "train_history": {"epochs": len(history["val_auc"]), "best_epoch": int(np.argmax(history["val_auc"])) + 1, "best_val_auc": float(max(history["val_auc"]))},
    "figures": {k: f"figures/{k}.png" for k in ["training_curves", "confusion", "roc_pr", "reliability", "ood_per_generator", "attention_rollout"]},
}
Me.save_metrics(record, dirs["metrics"] / "metrics.json")
print("saved", dirs["metrics"] / "metrics.json")
print("\nin-dist @0.5:  acc {accuracy:.4f}  f1 {f1_macro:.4f}  auc {auc_roc:.4f}  mcc {mcc:.4f}  brier {brier:.4f}".format(**m05))

**Next:** `08_clip-probe.ipynb` (frozen CLIP embeddings + trained head).